In [ ]:
!pip install -U bitsandbytes>=0.46.1
!pip install -U transformers datasets peft accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

def main():
    # Define the paths to your JSONL files
    train_path = r"/content/eval.jsonl"
    eval_path = r"/content/train.jsonl"

    # Ensure the files exist
    if not os.path.exists(train_path):
        print(f"Error: Train file not found at {train_path}")
        return
    if not os.path.exists(eval_path):
        print(f"Error: Eval file not found at {eval_path}")
        return

    # Define the data files mapping
    data_files = {
        "train": train_path,
        "validation": eval_path
    }

    print("Loading dataset...")
    # Load dataset using Hugging Face datasets library
    dataset = load_dataset("json", data_files=data_files)

    # Print dataset structure to verify
    print("\nDataset successfully loaded!")
    print(dataset)

    # Example of accessing an item
    if len(dataset["train"]) > 0:
        print("\nExample item from train split:")
        print(dataset["train"][0])

    # Optional: Save it to disk using Hugging Face format
    save_path = r"/content"
    dataset.save_to_disk(save_path)
    print(f"\nSaved dataset to {save_path}")

    return dataset

def setup_model(model_id: str, quantization_enabled: bool = True, peft_enabled: bool = True):
    print("\n--- MODEL LOADING PIPELINE ---")
    print(f"Base Model: HuggingFace Hub ({model_id})")

    # 1. Quantization Check
    if quantization_enabled:
        print("-> Quantization Enabled? Yes -> QLoRA")
        # BitsAndBytesConfig 4-bit NF4
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )

        print("-> AutoModelForCausalLM (Quantized)")
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto"
        )
    else:
        print("-> Quantization Enabled? No -> LoRA / Full Precision Model")
        print("-> AutoModelForCausalLM (Full Precision)")
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto"
        )

    # 2. PEFT Check
    if peft_enabled:
        print("-> PEFT Enabled? Yes")
        print("-> LoraConfig + get_peft_model")
        peft_config = LoraConfig(
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )
        model = get_peft_model(model, peft_config)
        model.print_trainable_parameters()
    else:
        print("-> PEFT Enabled? No -> Full Model Training")

    return model

def setup_training(model, tokenizer, dataset, training_method="SFT", output_dir="./results"):
    print("\n--- TRAINING PIPELINE ---")
    print(f"Training Method selected: {training_method}")

    if training_method == "SFT":
        from trl import SFTTrainer, SFTConfig
        print("-> Initializing SFTTrainer...")

        training_args = SFTConfig(
            output_dir=output_dir,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            learning_rate=2e-4,
            logging_steps=10,
            max_length=512,
            dataset_text_field="text", # Adjust based on your dataset format
            num_train_epochs=3
            #use_cpu=True
        )

        trainer = SFTTrainer(
            model=model,
            train_dataset=dataset["train"],
            eval_dataset=dataset.get("validation", None),
            args=training_args,
            processing_class=tokenizer,
        )

    elif training_method == "DPO":
        from trl import DPOTrainer, DPOConfig
        print("-> Initializing DPOTrainer...")

        training_args = DPOConfig(
            output_dir=output_dir,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=8,
            learning_rate=5e-5,
            logging_steps=10,
            beta=0.1,
        )

        # Note: DPO requires 'prompt', 'chosen', and 'rejected' columns in the dataset!
        trainer = DPOTrainer(
            model=model,
            train_dataset=dataset["train"],
            eval_dataset=dataset.get("validation", None),
            args=training_args,
            tokenizer=tokenizer,
        )

    elif training_method == "ORPO":
        from trl import ORPOTrainer, ORPOConfig
        print("-> Initializing ORPOTrainer...")

        training_args = ORPOConfig(
            output_dir=output_dir,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=8,
            learning_rate=5e-5,
            logging_steps=10,
            beta=0.1,
        )

        # Note: ORPO requires 'prompt', 'chosen', and 'rejected' columns
        trainer = ORPOTrainer(
            model=model,
            train_dataset=dataset["train"],
            eval_dataset=dataset.get("validation", None),
            tokenizer=tokenizer,
            args=training_args,
        )

    else:
        raise ValueError("Invalid training method. Choose SFT, DPO, or ORPO.")

    return trainer


def run_test():
    dataset = main()
    if not dataset:
        print("Dataset failed to load. Halting test.")
        return

    print("\n--- PREPARING TEST ENV ---")
    # Tiny model for fast sanity testing
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token

    # Map Alpaca formatting into a single 'text' column for SFTTrainer
    print("Formatting dataset to SFT expectations...")
    def map_to_text(example):
        if example.get('input'):
            text = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
        else:
            text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
        return {"text": text}

    dataset = dataset.map(map_to_text)

    # 1. Setup Model
    model = setup_model(model_id, quantization_enabled=True, peft_enabled=True)

    # 2. Setup Training
    trainer = setup_training(model, tokenizer, dataset, training_method="SFT", output_dir="./test_results")

    # Override max_steps for a quick test
    # trainer.args.max_steps = 1

    print("\nStarting the training run...")
    trainer.train()

    trainer.save_model("/content/final_model")
    tokenizer.save_pretrained("/content/final_model")
    print("\nModel saved to /content/final_model")


    print("\n--- TRAINING COMPLETED SUCCESSFULLY ---")


if __name__ == "__main__":
    run_test()

Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]


Dataset successfully loaded!
DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 291
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1643
    })
})

Example item from train split:
{'instruction': 'Illustrate the practical application of subsection (1) under Section 14 of THE CO-OPERATIVE SOCIETIES ACT, 1912 (1912).', 'input': 'The transfer or charge of the share or interest of a member in the capital of a registered society shall be subject to such conditions as to maximum holding as may be prescribed by this Act or by the rules.', 'output': 'This section would be invoked in a scenario where the transfer or charge of the share or interest of a member in the capital of a registered society shall be subject to such conditions as to maximum holding as may be prescribed by this Act or by the rules.'}


Saving the dataset (0/1 shards):   0%|          | 0/291 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/1643 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.



Starting the training run...


Step,Training Loss
10,1.908417
20,1.588216
30,1.409672
40,1.321655
50,1.184085



Model saved to /content/final_model

--- TRAINING COMPLETED SUCCESSFULLY ---


TIll Above is the Training and Saving the model part ,
Now from below we will load and test the model


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Base model
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("./final_model")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(base_model_id)

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, "./final_model")

model.eval()



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [ ]:
prompt = """### Instruction:
Explain subsection (2) under Section 15 of THE TEZPUR UNIVERSITY ACT, 1993.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

### Instruction:
Explain subsection (2) under Section 15 of THE TEZPUR UNIVERSITY ACT, 1993.

### Response:
(2) The Director shall not, without the previous approval of the Board, appoint any employee of the University as a member of any committee or body of the University.
